# Import Libraries

In [ ]:
!pip install torch_geometric -q

In [ ]:
!pip install igraph -q
from igraph import Graph

In [ ]:
import torch
from torch_geometric.datasets import AMiner, Taobao, MovieLens1M, AmazonBook, HM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import time
import psutil
import gc
import sys

In [ ]:
!git clone https://github.com/yutengkai/Louvain-Algorithm-Without-Graph-Construction.git

Cloning into 'Louvain-Algorithm-Without-Graph-Construction'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 29 (delta 8), reused 28 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 67.00 KiB | 6.70 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [ ]:
import os
if os.path.exists('Louvain-Algorithm-Without-Graph-Construction') and not os.path.exists('my_louvain'):
    os.rename('Louvain-Algorithm-Without-Graph-Construction', 'my_louvain')

In [ ]:
from my_louvain.src.data.data_downloading import *
from my_louvain.src.data.data_preprocessing import *
from my_louvain.src.main_algorithm import *
from my_louvain.src.utils import *

# Download and Preprocess Data

## AMiner

In [ ]:
AMiner_data = AMiner(root='data/AMiner')

Extracting data/AMiner/net_aminer.zip
Extracting data/AMiner/raw/label.zip
Processing...
Done!
/usr/local/lib/python3.10/dist-packages/torch_geometric/io/fs.py:215: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues re

In [ ]:
a_to_p = AMiner_data.data['author', 'writes', 'paper']['edge_index']
p_to_v = AMiner_data.data['paper', 'published_in', 'venue']['edge_index']

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
def join_relationships_count(type_a_to_intermediate_tensor, intermediate_to_type_b_tensor, type_a_name='type_A', type_b_name='type_B'):
    """
    Joins two torch tensors using an intermediate entity as the intermediary and counts the occurrences of each pair.

    Parameters:
    - type_a_to_intermediate_tensor (torch.Tensor): A tensor with shape [2, n] representing relationships between type A and intermediate.
    - intermediate_to_type_b_tensor (torch.Tensor): A tensor with shape [2, m] representing relationships between intermediate and type B.
    - type_a_name (str): Name of the first type (e.g., 'type_A').
    - type_b_name (str): Name of the second type (e.g., 'type_B').

    Returns:
    - pd.DataFrame: A DataFrame with columns 'type_A', 'type_B', and 'count'.
    """
    # Extract rows for easier understanding
    type_a_ids = type_a_to_intermediate_tensor[0]  # IDs for type A (e.g., authors)
    intermediate_ids_from_type_a = type_a_to_intermediate_tensor[1]  # intermediate IDs related to type A

    intermediate_ids_from_type_b = intermediate_to_type_b_tensor[0]  # intermediate IDs related to type B
    type_b_ids = intermediate_to_type_b_tensor[1]  # IDs for type B (e.g., venues)

    # Convert tensors to lists for easier manipulation
    type_a_ids = type_a_ids.tolist()
    intermediate_ids_from_type_a = intermediate_ids_from_type_a.tolist()
    intermediate_ids_from_type_b = intermediate_ids_from_type_b.tolist()
    type_b_ids = type_b_ids.tolist()

    # Create a dictionary that maps intermediate IDs to type B
    intermediate_to_type_b = {}
    for intermediate_id, type_b_id in zip(intermediate_ids_from_type_b, type_b_ids):
        if intermediate_id not in intermediate_to_type_b:
            intermediate_to_type_b[intermediate_id] = []
        intermediate_to_type_b[intermediate_id].append(type_b_id)

    # Create the joined data for type A and type B via intermediate
    joined_data = []
    for type_a_id, intermediate_id in zip(type_a_ids, intermediate_ids_from_type_a):
        if intermediate_id in intermediate_to_type_b:
            for type_b_id in intermediate_to_type_b[intermediate_id]:
                joined_data.append((type_a_id, type_b_id))

    # Convert to a DataFrame and count occurrences
    df = pd.DataFrame(joined_data, columns=[type_a_name, type_b_name])
    count_df = df.groupby([type_a_name, type_b_name]).size().reset_index(name='count')

    return count_df


In [ ]:
a_to_v_df = join_relationships_count(a_to_p, p_to_v)

In [ ]:
a_to_v_df.type_B.nunique()

3883

In [ ]:
a_to_v_df[a_to_v_df['count'] > 100].type_B.nunique()

119

In [ ]:
def df_to_matrix(df):
    [c1, c2, c3] = df.columns.tolist()
    pivot_df = df.pivot(index=c1, columns=c2, values=c3)

    pivot_df.fillna(0, inplace=True)

    return pivot_df

In [ ]:
def plot_grouped_distribution(df):
    """
    Group the DataFrame by the second column, aggregate the third column (count), and plot the distribution.

    Parameters:
    - df (pd.DataFrame): The input DataFrame with three columns.
    """
    # Assuming the second column is the one to group by, and the third column is the count
    second_column = df.columns[1]  # Second column (e.g., type_B)
    third_column = df.columns[2]   # Third column (e.g., count)

    # Group by the second column and sum the third column
    aggregated_df = df.groupby(second_column)[third_column].sum().reset_index()

    # Plot the distribution of the aggregated counts
    plt.figure(figsize=(10, 6))

    sns.histplot(aggregated_df[third_column], bins=30, kde=True, color='blue')

    plt.title(f'Distribution of Aggregated Values by {second_column}')
    plt.xlabel('Aggregated Count')
    plt.ylabel('Frequency')

    plt.grid(True)
    plt.show()

# Assuming df is your DataFrame
# plot_grouped_distribution(a_to_v_df)


In [ ]:
def filter_by_threshold(df, threshold_percentage=1, column_to_filter=1):
    """
    Filter the DataFrame based on a count threshold applied to either the first or second column.

    Parameters:
    - df (pd.DataFrame): The input DataFrame with at least two columns.
    - threshold_percentage (float): The threshold percentage of the maximum count.
    - filter_on (str): Specify 'first' to filter based on the first column, or 'second' to filter based on the second column. Default is 'second'.

    Returns:
    - pd.DataFrame: A filtered DataFrame where entries in the specified column meet the count threshold.
    """

    # Calculate the total number of unique values in the selected column
    col_counts = df.iloc[:, column_to_filter].value_counts()

    # Calculate the maximum count (the most frequent value in the selected column)
    max_count = col_counts.max()
    print(max_count)
    # Calculate the threshold count
    threshold = max_count * (threshold_percentage / 100)

    # Filter the DataFrame based on the threshold
    filtered_values = col_counts[col_counts >= threshold].index
    filtered_df = df[df.iloc[:, column_to_filter].isin(filtered_values)]

    return filtered_df



In [ ]:
filtered_a_to_v_df = filter_by_threshold(a_to_v_df, 10)
filtered_a_to_v_df = filter_by_threshold(filtered_a_to_v_df, 6, 0)

34028
184


In [ ]:
filtered_a_to_v_df.shape

(392682, 3)

In [ ]:
# plot_grouped_distribution(filtered_a_to_v_df)

In [ ]:
filtered_a_to_v_df.nunique()

,0
type_A,20103
type_B,337
count,120


In [ ]:
pivot_df = df_to_matrix(filtered_a_to_v_df)

In [ ]:
pivot_df.shape

(20103, 337)

In [ ]:
def convert_to_tfidf(pivot_df):

    # Step 2: Apply TF-IDF transformation
    tfidf_transformer = TfidfTransformer()
    tfidf_matrix = tfidf_transformer.fit_transform(pivot_df)

    # Step 3: Convert the resulting matrix back to a DataFrame for easier analysis
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=pivot_df.index, columns=pivot_df.columns)

    return tfidf_df

In [ ]:
# tfidf_df = convert_to_tfidf(pivot_df)
# adj = cosine_similarity(tfidf_df)
# np.fill_diagonal(adj, 0)
# adj
# ig = Graph.Weighted_Adjacency(adj, mode='undirected')
# ig_partition = ig.community_multilevel(weights='weight')
# ig.modularity(ig_partition.membership, weights='weight')
# len(ig_partition)
# tfidf_matrix = torch.tensor(tfidf_df.values)
# my_partitions = louvain_partition_pos_neg(tfidf_matrix.to('cuda'), gamma=1.0, threshold=1e-7, max_level=-1, seed=42)
# final_communities = get_final_communities(my_partitions)
# ig.modularity(final_communities, weights = 'weight')


In [ ]:
def count_edges(tfidf_matrix):
    """
    Counts how many inner products between rows of the TF-IDF matrix are greater than zero.

    Parameters:
    - tfidf_matrix (torch.Tensor): The input TF-IDF matrix, assumed to be a PyTorch tensor.

    Returns:
    - List[int]: A list where each element corresponds to the count of inner products greater than zero
                 for each row when compared with the remaining rows.
    """
    # Move the TF-IDF matrix to CUDA
    tfidf_matrix_cuda = tfidf_matrix.to('cuda')

    # Initialize the result list
    counts = []

    # Iterate over each row
    for i in range(tfidf_matrix_cuda.shape[0] - 1):
        # Get the current row
        current_row = tfidf_matrix_cuda[i]

        # Perform the inner product with the remaining rows (i+1 to the end)
        remaining_rows = tfidf_matrix_cuda[i + 1:]

        # Compute the inner product
        inner_product = torch.matmul(remaining_rows, current_row)

        # Count how many inner products are greater than zero
        count_above_zero = torch.sum(inner_product > 0).item()

        # Append the count to the result list
        counts.append(count_above_zero)

    return sum(counts)

In [ ]:
def run_experiments(df, threshold_list, filter_column=1):
    """
    Runs experiments for I-graph and Graph-less Louvain algorithms and records results.

    Parameters:
    - df (pd.DataFrame): The input DataFrame.
    - threshold_list (list): List of threshold percentages to filter on.
    - filter_column (int): Index of the column to filter on (0 for first column, 1 for second column).
    - max_memory_gb (float): The maximum available memory in GB for I-graph execution.

    Returns:
    - pd.DataFrame: DataFrame with experiment results for each threshold.
    """
    results = []

    # Iterate through each threshold
    for threshold in threshold_list:
        total_memory_gb = psutil.virtual_memory().available / (1024 ** 3)
        row_result = {'threshold': threshold}

        # Step 1: Filter the DataFrame
        filtered_df = filter_by_threshold(df, threshold, filter_column)
        num_type_a = filtered_df.iloc[:, 0].nunique()  # Number of type A nodes
        num_type_b = filtered_df.iloc[:, 1].nunique()  # Number of type B nodes
        num_edges = filtered_df.shape[0]  # Number of edges (heterogeneous edges)

        row_result['num_type_a'] = num_type_a
        row_result['num_type_b'] = num_type_b
        row_result['num_hetero_edges'] = num_edges

        # Step 2: Pivot the DataFrame
        pivot_df = df_to_matrix(filtered_df)

        # Estimate memory cost for I-graph
        estimated_memory_gb = (pivot_df.shape[0] ** 2) * (40 / (20000 ** 2))  # Memory cost estimate in GB

        # Step 3: Calculate TF-IDF
        tfidf_df = convert_to_tfidf(pivot_df)

        # if estimated_memory_gb > total_memory_gb:
        if pivot_df.shape[0] > 25000:
            # Skip I-graph if estimated memory exceeds available or maximum memory
            row_result['igraph_memory'] = 0
            row_result['igraph_time'] = 0
            row_result['igraph_modularity'] = 0
            row_result['igraph_edges'] = 0
        else:
             # Step 4: Run I-graph Partitioning
            start_time = time.time()
            adj = cosine_similarity(tfidf_df)  # Cosine similarity for I-graph
            np.fill_diagonal(adj, 0)  # Set diagonal to 0
            ig = Graph.Weighted_Adjacency(adj, mode='undirected')  # Create I-graph object
            ig_partition = ig.community_multilevel(weights='weight')  # Run Louvain partition
            ig_modularity = ig.modularity(ig_partition.membership, weights='weight')  # Modularity
            num_partitions = len(ig_partition)  # Number of partitions
            ig_time = time.time() - start_time  # Time taken
            ig_ecount = ig.ecount()  # Number of edges

            # Record I-graph results
            row_result['igraph_memory'] = (sys.getsizeof(adj) + sys.getsizeof(ig) + sys.getsizeof(ig_partition)) / 1024**3
            row_result['igraph_time'] = ig_time
            row_result['igraph_modularity'] = ig_modularity
            row_result['igraph_edges'] = ig_ecount

            # Free up I-graph memory
            del ig, ig_partition, adj
            gc.collect()

        # ---------------------- Graph-less Louvain Part ----------------------
        # Step 5: Run your Graph-less Louvain (GPU-based)
        start_time = time.time()
        tfidf_matrix = torch.tensor(tfidf_df.values)
        my_partitions = louvain_partition_gpu(tfidf_matrix.to('cuda'), gamma=1.0, threshold=1e-7, max_level=-1, seed=42)
        final_communities = get_final_communities(my_partitions)
        graphless_modularity = modularity_all_partitions(tfidf_matrix, final_communities.to('cpu'))
        graphless_time = time.time() - start_time  # Time taken

        # Record GPU memory usage (if possible in Colab)
        gpu_memory = torch.cuda.memory_allocated() / (1024 ** 3)  # Memory usage in GB

        # Record Graph-less Louvain results
        row_result['num_partitions'] = len(np.unique(final_communities.to('cpu')))
        row_result['graphless_time'] = graphless_time
        row_result['gpu_memory'] = gpu_memory
        row_result['graphless_modularity'] = graphless_modularity
        row_result['graphless_edges'] = count_edges(tfidf_matrix)
        # Append row to results
        results.append(row_result)

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    return results_df

In [ ]:
filtered_a_to_v_df = filter_by_threshold(a_to_v_df, 10)
run_experiments(filtered_a_to_v_df, [20, 15, 10, 5], 0)

34028
184


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 1019.82it/s]


184


Processing nodes: 100%|██████████| 3/3 [00:00<00:00, 883.20it/s]


184


Processing nodes: 100%|██████████| 3/3 [00:00<00:00, 1032.83it/s]


184


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 1121.83it/s]


,threshold,num_type_a,num_type_b,num_hetero_edges,igraph_memory,igraph_time,igraph_modularity,igraph_edges,num_partitions,graphless_time,gpu_memory,graphless_modularity,graphless_edges
0,20,1098,337,65810,0.008983,0.555745,0.085662,601778,5,1.645899,0.007992,0.082453,601778
1,15,2381,337,105821,0.042239,4.914661,0.153029,2789744,3,3.490510,0.008010,0.147548,2789744
2,10,6795,337,203053,0.344009,45.915958,0.231854,20340414,3,10.368363,0.008076,0.228562,20340414
3,5,29366,337,489409,0.000000,0.000000,0.000000,0,5,44.791336,0.008412,0.308657,231873312


In [ ]:
filtered_a_to_v_df = filter_by_threshold(a_to_v_df, 5)
run_experiments(filtered_a_to_v_df, [20, 15, 10, 5], 0)

34028
267


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 997.12it/s]


267


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 1194.82it/s]


267


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 1154.31it/s]


267


Processing nodes: 100%|██████████| 6/6 [00:00<00:00, 835.02it/s]


,threshold,num_type_a,num_type_b,num_hetero_edges,igraph_memory,igraph_time,igraph_modularity,igraph_edges,num_partitions,graphless_time,gpu_memory,graphless_modularity,graphless_edges
0,20,1096,793,96370,0.008950,0.620064,0.089511,599101,5,1.715569,0.007991,0.085442,599101
1,15,2370,793,154644,0.041850,3.902420,0.153570,2740373,5,3.525639,0.008011,0.147635,2740373
2,10,7413,793,315238,0.409429,62.706871,0.248746,23014968,5,12.591184,0.008085,0.241160,23014968
3,5,31653,793,754879,0.000000,0.000000,0.000000,0,6,53.023631,0.008447,0.330396,250266599


In [ ]:
torch.cuda.memory_allocated()

8560640

In [ ]:
psutil.virtual_memory().available / (1024 ** 3)

72.04304885864258

# MovieLens1M

In [ ]:
MovieLens1M_data = MovieLens1M(root='data/MovieLens1M')

Extracting data/MovieLens1M/ml-1m.zip
Processing...
Done!
/usr/local/lib/python3.10/dist-packages/torch_geometric/io/fs.py:215: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

In [ ]:
ratings_tensor = MovieLens1M_data.data['user', 'rates', 'movie']['rating']

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
user_movie_tensor = MovieLens1M_data.data['user', 'rates', 'movie']['edge_index']

In [ ]:
user_movie_tensor = user_movie_tensor.T
user_movie_df = pd.DataFrame({
        'user': user_movie_tensor[:, 0].numpy(),
        'movie': user_movie_tensor[:, 1].numpy(),
        'rating': ratings_tensor.numpy()
    })
user_movie_df.head()

,user,movie,rating
0,0,1176,5
1,0,655,3
2,0,902,3
3,0,3339,4
4,0,2286,5


In [ ]:
run_experiments(user_movie_df, [20, 15, 10, 5, 0], 0)

2314


Processing nodes: 100%|██████████| 3/3 [00:00<00:00, 1046.66it/s]


2314


Processing nodes: 100%|██████████| 3/3 [00:00<00:00, 1037.94it/s]


2314


...

2314


Processing nodes: 100%|██████████| 3/3 [00:00<00:00, 986.04it/s]


2314


Processing nodes: 100%|██████████| 4/4 [00:00<00:00, 1079.41it/s]


,threshold,num_type_a,num_type_b,num_hetero_edges,igraph_memory,igraph_time,igraph_modularity,igraph_edges,num_partitions,graphless_time,gpu_memory,graphless_modularity,graphless_edges
0,20,458,3613,318693,0.001563,0.121755,0.047188,104653,3,0.688964,0.007981,0.048204,104653
1,15,763,3634,440410,0.004338,0.315844,0.050654,290703,3,1.126614,0.007985,0.049551,290703
2,10,1329,3655,601253,0.013160,1.043661,0.059235,882451,3,2.046337,0.007994,0.057279,882451
3,5,2619,3670,812647,0.051105,5.229897,0.072652,3427576,3,3.938659,0.008013,0.069276,3427576
4,0,6040,3706,1000209,0.271809,54.601369,0.091654,17472510,4,11.065673,0.008065,0.088371,17472510


In [ ]:
run_experiments(user_movie_df, [20, 15, 10, 5, 0], 1)

3428


Processing nodes: 100%|██████████| 4/4 [00:00<00:00, 1030.98it/s]


3428


Processing nodes: 100%|██████████| 3/3 [00:00<00:00, 1029.95it/s]


3428


...

3428


Processing nodes: 100%|██████████| 4/4 [00:00<00:00, 993.85it/s]


3428


Processing nodes: 100%|██████████| 4/4 [00:00<00:00, 1088.02it/s]


,threshold,num_type_a,num_type_b,num_hetero_edges,igraph_memory,igraph_time,igraph_modularity,igraph_edges,num_partitions,graphless_time,gpu_memory,graphless_modularity,graphless_edges
0,20,6039,411,472869,0.271720,30.146841,0.078562,17334980,4,9.943370,0.008064,0.070416,17334980
1,15,6040,598,584564,0.271809,30.792207,0.082031,17394944,3,9.992409,0.008064,0.077970,17394944
2,10,6040,951,731733,0.271809,52.565013,0.087355,17437907,4,9.173841,0.008064,0.083803,17437907
3,5,6040,1563,881170,0.271809,37.763790,0.092291,17464350,4,10.190469,0.008065,0.084919,17464350
4,0,6040,3706,1000209,0.271809,48.012585,0.091654,17472510,4,11.166602,0.008065,0.088371,17472510


# Taobao

In [ ]:
Taobao_data = Taobao(root='data/Taobao')

In [ ]:
u_t_i = Taobao_data.data['user', 'to', 'item']['edge_index']
i_t_c = Taobao_data.data['item', 'to', 'category']['edge_index']

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
taobao_df = join_relationships_count(u_t_i, i_t_c)

In [ ]:
taobao_df.shape

(24211867, 3)

In [ ]:
taobao_df.type_B.nunique()

9437

In [ ]:
filtered_taobao_df = filter_by_threshold(taobao_df, 15)

417880


In [ ]:
filtered_taobao_df.shape

(9515520, 3)

In [ ]:
filtered_taobao_df.type_B.nunique()

66

In [ ]:
run_experiments(filtered_taobao_df, [55, 50, 45], 0)

54


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 1149.06it/s]


54


Processing nodes: 100%|██████████| 4/4 [00:00<00:00, 915.84it/s]


54


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 1101.68it/s]


,threshold,num_type_a,num_type_b,num_hetero_edges,igraph_memory,igraph_time,igraph_modularity,igraph_edges,num_partitions,graphless_time,gpu_memory,graphless_modularity,graphless_edges
0,55,8404,66,272136,0.526214,107.343935,0.05904,35309406,5,15.179235,0.008062,0.058624,35309406
1,50,20663,66,613264,3.181097,550.458143,0.06495,213469453,4,41.218935,0.008244,0.062313,213469453
2,45,34859,66,974454,0.000000,0.000000,0.00000,0,5,54.137748,0.008456,0.069337,607557511


In [ ]:
run_experiments(filtered_taobao_df, [40, 35, 30], 0)

54


Processing nodes: 100%|██████████| 6/6 [00:00<00:00, 1073.12it/s]


54


Processing nodes: 100%|██████████| 5/5 [00:00<00:00, 1063.46it/s]


54


Processing nodes: 100%|██████████| 6/6 [00:00<00:00, 1125.08it/s]


,threshold,num_type_a,num_type_b,num_hetero_edges,igraph_memory,igraph_time,igraph_modularity,igraph_edges,num_partitions,graphless_time,gpu_memory,graphless_modularity,graphless_edges
0,40,70696,66,1794417,0,0,0,0,6,108.606565,0.008990,0.075408,2498926621
1,35,126927,66,2914204,0,0,0,0,5,202.765124,0.009828,0.082280,8055086840
2,30,177836,66,3803782,0,0,0,0,6,281.597357,0.010586,0.088723,15811491792


In [ ]:
run_experiments(filtered_taobao_df, [25, 20, 15], 0)

54


Processing nodes: 100%|██████████| 6/6 [00:00<00:00, 1181.44it/s]


54


Processing nodes: 100%|██████████| 6/6 [00:00<00:00, 1100.34it/s]


54


Processing nodes: 100%|██████████| 6/6 [00:00<00:00, 1028.14it/s]


,threshold,num_type_a,num_type_b,num_hetero_edges,igraph_memory,igraph_time,igraph_modularity,igraph_edges,num_partitions,graphless_time,gpu_memory,graphless_modularity,graphless_edges
0,25,273616,66,5235239,0,0,0,0,6,445.907921,0.012014,0.098530,37399121427
1,20,393051,66,6663332,0,0,0,0,6,652.646971,0.013793,0.107986,76677336152
2,15,485086,66,7536260,0,0,0,0,6,836.071086,0.015165,0.123505,114973204649


In [ ]:
# run_experiments(filtered_taobao_df, [10, 5, 0], 0)